# NutriScan AI - Entrenamiento del Modelo (v2)

Clasificador de 10 alimentos usando Transfer Learning con **EfficientNetB0**.

Mejoras respecto a v1:
- EfficientNetB0 (mayor precisión que MobileNetV2)
- Data Augmentation integrada como capas del modelo (ejecución en GPU)
- Entrenamiento en dos fases: Feature Extraction + Fine-Tuning
- Callbacks: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

Dataset: Food-101 (subconjunto de 10 clases)

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json

print('TensorFlow version:', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))

## 1. Configuración

EfficientNetB0 usa entrada 224×224×3 (compatible con el preprocesamiento actual).

Estructura esperada del dataset:
```
data/
  train/
    apple_pie/
    hamburger/
    ...
  test/
    apple_pie/
    hamburger/
    ...
```

In [ ]:
CLASSES = [
    'apple_pie', 'hamburger', 'french_fries', 'hot_dog', 'sushi',
    'donuts', 'ice_cream', 'chicken_wings', 'pizza', 'caesar_salad'
]
NUM_CLASSES = len(CLASSES)
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_FE = 15     # Feature Extraction
EPOCHS_FT = 20     # Fine-Tuning
DATA_DIR = '../data'

print(f'Clases ({NUM_CLASSES}): {CLASSES}')

In [ ]:
train_ds = image_dataset_from_directory(
    os.path.join(DATA_DIR, 'train'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

test_ds = image_dataset_from_directory(
    os.path.join(DATA_DIR, 'test'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

print(f'Train batches: {len(train_ds)}, Test batches: {len(test_ds)}')

## 2. Normalización del dataset

Solo convertimos a float32; EfficientNetB0 tiene su propio `preprocess_input` interno.

In [ ]:
def convert_to_float(image, label):
    image = tf.cast(image, tf.float32)
    return image, label

train_ds = train_ds.map(convert_to_float).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(convert_to_float).prefetch(tf.data.AUTOTUNE)

## 3. Arquitectura del modelo (EfficientNetB0 + Data Augmentation interna)

Data Augmentation se coloca **dentro del modelo** como capas de Keras:
Esto permite que las transformaciones se ejecuten en GPU durante el entrenamiento
y se desactiven automáticamente en inferencia.

In [ ]:
def build_model():
    # --- Data Augmentation (solo activa durante training) ---
    data_augmentation = keras.Sequential([
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.2),
        layers.RandomZoom(0.2),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomContrast(0.15),
    ], name='data_augmentation')

    # --- Base Model: EfficientNetB0 ---
    base_model = keras.applications.EfficientNetB0(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False  # Feature Extraction

    # --- Construcción ---
    inputs = keras.Input(shape=(224, 224, 3))
    x = data_augmentation(inputs)
    x = keras.applications.efficientnet.preprocess_input(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = models.Model(inputs, outputs, name='NutriScan_EfficientNetB0')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model, base_model

model, base_model = build_model()
model.summary()

## 4. Fase 1 - Feature Extraction

Solo se entrena la cabeza clasificadora. El backbone EfficientNetB0 permanece congelado.

In [ ]:
callbacks_fe = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6
    ),
    keras.callbacks.ModelCheckpoint(
        '../model/nutriscan_fe.keras',
        monitor='val_accuracy', save_best_only=True
    )
]

history_fe = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS_FE,
    callbacks=callbacks_fe
)

## 5. Fase 2 - Fine-Tuning

Se descongela EfficientNetB0 y se entrena con un learning rate menor.
Esto permite ajustar los pesos del backbone a nuestro dominio específico.

In [ ]:
base_model.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_ft = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=7, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7
    ),
    keras.callbacks.ModelCheckpoint(
        '../model/nutriscan_model.keras',
        monitor='val_accuracy', save_best_only=True
    )
]

history_ft = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS_FE + EPOCHS_FT,
    initial_epoch=len(history_fe.history['loss']),
    callbacks=callbacks_ft
)

## 6. Evaluación del modelo final

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f'Test accuracy: {test_acc:.4f}')
print(f'Test loss: {test_loss:.4f}')

In [ ]:
y_true = []
y_pred = []
y_probs = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))
    y_probs.extend(np.max(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_probs = np.array(y_probs)

print('\n=== Classification Report ===')
print(classification_report(y_true, y_pred, target_names=CLASSES))

print(f'\nConfianza promedio en aciertos: {np.mean(y_probs[y_true == y_pred]):.4f}')
print(f'Confianza promedio en errores: {np.mean(y_probs[y_true != y_pred]):.4f}')

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.title('Matriz de Confusión - NutriScan AI (EfficientNetB0)')
plt.tight_layout()

os.makedirs('../report/images', exist_ok=True)
plt.savefig('../report/images/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Guardar modelo final

El modelo final se guarda en `model/nutriscan_model.keras` para ser usado por la app Streamlit.

In [ ]:
os.makedirs('../model', exist_ok=True)
model.save('../model/nutriscan_model.keras')
print('Modelo final guardado en model/nutriscan_model.keras')

## 8. (Opcional) Expandir clases

Para agregar nuevas clases:
1. Agrega el nombre al array `CLASSES` arriba.
2. Agrega las imágenes en `data/train/<nueva_clase>/` y `data/test/<nueva_clase>/`.
3. Actualiza `labels.json` con el nuevo mapeo.
4. Agrega las calorías en `calories.json`.
5. Vuelve a ejecutar todo el notebook.

```python
# Ejemplo: agregar 'nachos'
CLASSES = [
    'apple_pie', 'hamburger', 'french_fries', 'hot_dog', 'sushi',
    'donuts', 'ice_cream', 'chicken_wings', 'pizza', 'caesar_salad',
    'nachos'  # nuevo
]
```